In [0]:
import requests
from pyspark.sql.functions import current_timestamp

In [0]:
dbutils.widgets.text("API_KEY", "")
dbutils.widgets.text("SUBGRAPH_ID", "")

In [0]:
API_KEY = dbutils.widgets.get("API_KEY")
SUBGRAPH_ID = dbutils.widgets.get("SUBGRAPH_ID")

In [0]:
SUBGRAPH_UNIV3_URL = f"https://gateway.thegraph.com/api/{API_KEY}/subgraphs/id/{SUBGRAPH_ID}"

In [0]:
token_query = """
{
  tokens(first: 1000, orderBy: volumeUSD, orderDirection: desc) {
    id
    symbol
    name
    decimals
    totalSupply
    volume
    volumeUSD
    feesUSD
    poolCount
    totalValueLocked
    totalValueLockedUSD
    whitelistPools
  }
}
"""

In [0]:
response = requests.post(SUBGRAPH_UNIV3_URL, json={"query": token_query})

In [0]:
df = spark.createDataFrame(response.json()["data"]["tokens"])
df = df.withColumn("_load_ts", current_timestamp())

In [0]:
display(df)

In [0]:
df.write.format("delta").mode("append").saveAsTable("uniswap.bronze.tokens")